In [0]:
# Import required libraries
from pyspark.sql import functions as F

# Azure SQL connection details
server_name = "capstone09.database.windows.net"
database_name = "capstone09"

jdbc_url = (
    f"jdbc:sqlserver://{server_name}:1433;"
    f"databaseName={database_name};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

# Get credentials securely
username = dbutils.secrets.get(
    scope="azure-sql-scope",
    key="sql-user"
)

password = dbutils.secrets.get(
    scope="azure-sql-scope",
    key="sql-pwd"
)

connection_properties = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print("JDBC configuration completed")

In [0]:
# Read meter master
meter_df = (
    spark.read
    .jdbc(
        url=jdbc_url,
        table="dbo.meter_master",
        properties=connection_properties
    )
)

# Read substation master
substation_df = (
    spark.read
    .jdbc(
        url=jdbc_url,
        table="dbo.substation_master",
        properties=connection_properties
    )
)

# Read tariff lookup
tariff_df = (
    spark.read
    .jdbc(
        url=jdbc_url,
        table="dbo.tariff_lookup",
        properties=connection_properties
    )
)

print("All three Azure SQL tables read successfully")

In [0]:
print("METER MASTER")
print(meter_df.count())
display(meter_df.limit(5))

print("SUBSTATION MASTER")
print(substation_df.count())
display(substation_df.limit(5))

print("TARIFF LOOKUP")
print(tariff_df.count())
display(tariff_df.limit(5))

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import *

# --------------------------------------------------
# 1. SELECT EXISTING CATALOG AND SCHEMA
# --------------------------------------------------

spark.sql("USE CATALOG capstone9_smartmeter")
spark.sql("USE SCHEMA smart_grid_bronze")

# --------------------------------------------------
# 2. DEFINE SOURCE, SCHEMA AND CHECKPOINT PATHS
# --------------------------------------------------

base_path = "/Volumes/capstone9_smartmeter/smart_grid_bronze/raw_volume"

schema_base = f"{base_path}/_schemas/smart_grid_new"
checkpoint_base = f"{base_path}/_checkpoints/smart_grid_new"

for folder in ["readings", "outages", "recovery"]:
    dbutils.fs.mkdirs(f"{schema_base}/{folder}")
    dbutils.fs.mkdirs(f"{checkpoint_base}/{folder}")

# --------------------------------------------------
# 3. DEFINE EXPLICIT SCHEMAS
# --------------------------------------------------

readings_schema = StructType([
    StructField("reading_id", StringType(), True),
    StructField("meter_id", StringType(), True),
    StructField("substation_id", StringType(), True),
    StructField("interval_start_ts", TimestampType(), True),
    StructField("interval_end_ts", TimestampType(), True),
    StructField("energy_kwh", DoubleType(), True),
    StructField("voltage_v", DoubleType(), True),
    StructField("current_a", DoubleType(), True),
    StructField("power_factor", DoubleType(), True),
    StructField("quality_code", StringType(), True),
    StructField("reading_source", StringType(), True),
    StructField("received_ts", TimestampType(), True)
])

outage_schema = StructType([
    StructField("outage_id", StringType(), True),
    StructField("substation_id", StringType(), True),
    StructField("outage_start_ts", TimestampType(), True),
    StructField("outage_end_ts", TimestampType(), True),
    StructField("outage_cause", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("affected_meters", IntegerType(), True),
    StructField("incident_status", StringType(), True),
    StructField("reported_ts", TimestampType(), True),
    StructField("repair_team", StringType(), True)
])

recovery_schema = StructType([
    StructField("recovery_event_id", StringType(), True),
    StructField("meter_id", StringType(), True),
    StructField("outage_id", StringType(), True),
    StructField("event_ts", TimestampType(), True),
    StructField("recovery_type", StringType(), True),
    StructField("signal_strength", DoubleType(), True),
    StructField("firmware_version", StringType(), True),
    StructField("acknowledged_flag", BooleanType(), True),
    StructField("technician_id", StringType(), True),
    StructField("received_ts", TimestampType(), True)
])

# --------------------------------------------------
# 4. AUTO LOADER - SMART METER READINGS
# --------------------------------------------------

readings_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{schema_base}/readings")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(readings_schema)
    .load(f"{base_path}/telemetry_data")
)

# --------------------------------------------------
# 5. AUTO LOADER - GRID OUTAGE EVENTS
# --------------------------------------------------

outage_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("pathGlobFilter", "grid_outage_events.csv")
    .option("cloudFiles.schemaLocation", f"{schema_base}/outages")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(outage_schema)
    .load(f"{base_path}/event_data")
)

# --------------------------------------------------
# 6. AUTO LOADER - METER RECOVERY EVENTS
# --------------------------------------------------

recovery_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("pathGlobFilter", "meter_recovery_events.csv")
    .option("cloudFiles.schemaLocation", f"{schema_base}/recovery")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(recovery_schema)
    .load(f"{base_path}/event_data")
)

# --------------------------------------------------
# 7. ADD INGESTION METADATA
# --------------------------------------------------

readings_df = (
    readings_df
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

outage_df = (
    outage_df
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

recovery_df = (
    recovery_df
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

# --------------------------------------------------
# 8. WRITE TO BRONZE DELTA TABLES
# --------------------------------------------------

query1 = (
    readings_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{checkpoint_base}/readings")
    .trigger(availableNow=True)
    .toTable("bronze_smart_meter_readings")
)

query2 = (
    outage_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{checkpoint_base}/outages")
    .trigger(availableNow=True)
    .toTable("bronze_grid_outage_events")
)

query3 = (
    recovery_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{checkpoint_base}/recovery")
    .trigger(availableNow=True)
    .toTable("bronze_meter_recovery_events")
)

# Wait for all availableNow streams to finish before validating
for q in [query1, query2, query3]:
    q.awaitTermination()

# --------------------------------------------------
# 9. VALIDATE BRONZE TABLES
# --------------------------------------------------

for table in [
    "bronze_smart_meter_readings",
    "bronze_grid_outage_events",
    "bronze_meter_recovery_events"
]:
    count = spark.table(table).count()
    print(f"{table}: {count} records")

print("ADLS AUTO LOADER BRONZE INGESTION COMPLETED")


In [0]:
spark.sql("""
SELECT 'Smart Meter Readings' AS dataset, COUNT(*) AS records
FROM capstone9_smartmeter.smart_grid_bronze.bronze_smart_meter_readings

UNION ALL

SELECT 'Grid Outage Events', COUNT(*)
FROM capstone9_smartmeter.smart_grid_bronze.bronze_grid_outage_events

UNION ALL

SELECT 'Meter Recovery Events', COUNT(*)
FROM capstone9_smartmeter.smart_grid_bronze.bronze_meter_recovery_events
""").show()

In [0]:

from pyspark.sql import functions as F

# 1. Read the three Azure SQL tables

meter_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.meter_master",
    properties=connection_properties
)

substation_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.substation_master",
    properties=connection_properties
)

tariff_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.tariff_lookup",
    properties=connection_properties
)

print("All three Azure SQL tables read successfully!")

# 2. Select existing catalog and schema

spark.sql("USE CATALOG capstone9_smartmeter")
spark.sql("USE SCHEMA smart_grid_bronze")

# 3. Add ingestion metadata and write Bronze Delta tables

datasets = [
    (meter_df, "bronze_meter_master"),
    (substation_df, "bronze_substation_master"),
    (tariff_df, "bronze_tariff_lookup")
]

for df, table_name in datasets:

    bronze_df = (
        df
        .withColumn("_source", F.lit("Azure SQL"))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )

    print(f"{table_name} created successfully")

# 4. Validate all three tables

for _, table_name in datasets:
    full_table_name = (
        f"capstone9_smartmeter.smart_grid_bronze.{table_name}"
    )

    print(
        table_name,
        "Records:",
        spark.table(full_table_name).count()
    )

print("AZURE SQL TO BRONZE INGESTION COMPLETED!")
